In [6]:
import json
from pathlib import Path
import pandas as pd

# Correct path pointing directly to experiments/day3/
json_path = Path("../experiments/day3/conformal_params_v2.json")

with open(json_path, "r") as f:
  conformal_params = json.load(f)

print("Conformal parameters loaded successfully from:", json_path)
print("Keys available:", list(conformal_params.keys()))

Conformal parameters loaded successfully from: ..\experiments\day3\conformal_params_v2.json
Keys available: ['model', 'method', 'groups', 'distance_edges_km', 'min_group_size', 'calibration_legs', 'levels']


In [8]:
import json
from pathlib import Path
import pandas as pd

# Load the saved XGBoost residual model artifact from models/
model_path = Path("../models/residual_xgb_plain_v1.json")

with open(model_path, "r") as f:
  model_data = json.load(f)

print("XGBoost residual model artifact loaded successfully from:", model_path)

XGBoost residual model artifact loaded successfully from: ..\models\residual_xgb_plain_v1.json


In [9]:
# Load conformal parameters from experiments/day3/
with open("../experiments/day3/conformal_params_v2.json", "r") as f:
  conformal_params = json.load(f)

# Load core network edges
core_edges = pd.read_csv("../data/processed/core_edges.csv")

print("Conformal parameters and core edges loaded successfully!")
print("Core edges shape:", core_edges.shape)
print("Core edges columns:", core_edges.columns.tolist())

Conformal parameters and core edges loaded successfully!
Core edges shape: (292, 13)
Core edges columns: ['source', 'destination', 'trips', 'median_actual_time', 'median_ratio', 'p90_ratio', 'distance_km', 'stats_legs', 'stats_cutoff_exclusive', 'source_center', 'destination_center', 'leg_count', 'trip_count']


In [10]:
import numpy as np

# Prepare a list to store risk table rows
risk_rows = []

# Iterate through each core edge to compute predictions & conformal bounds
for idx, row in core_edges.iterrows():
  dist = row["distance_km"]
  median_time = row["median_actual_time"]
  trip_count = row["trip_count"]
  leg_count = row["leg_count"]

  # Use core_edges median_ratio as our baseline prediction
  pred_ratio = row["median_ratio"]

  # Apply conformal margins from your loaded conformal_params
  margin_50 = conformal_params.get("levels", {}).get("0.5", 0.05)
  margin_90 = conformal_params.get("levels", {}).get("0.9", 0.15)

  lo50 = max(0.0, pred_ratio - margin_50)
  hi50 = pred_ratio + margin_50
  lo90 = max(0.0, pred_ratio - margin_90)
  hi90 = pred_ratio + margin_90

  pred_excess_h = (pred_ratio - 1.0) * median_time
  excess_lo90_h = (lo90 - 1.0) * median_time
  excess_hi90_h = (hi90 - 1.0) * median_time

  risk_rows.append(
      {
          "source": row["source"],
          "destination": row["destination"],
          "source_center": row["source_center"],
          "destination_center": row["destination_center"],
          "leg_count": leg_count,
          "trip_count": trip_count,
          "distance_km": dist,
          "median_actual_time": median_time,
          "pred_ratio": round(pred_ratio, 4),
          "lo50": round(lo50, 4),
          "hi50": round(hi50, 4),
          "lo90": round(lo90, 4),
          "hi90": round(hi90, 4),
          "pred_excess_h": round(pred_excess_h, 4),
          "excess_lo90_h": round(excess_lo90_h, 4),
          "excess_hi90_h": round(excess_hi90_h, 4),
      }
  )

risk_table = pd.DataFrame(risk_rows)
print("Risk table generated successfully! Shape:", risk_table.shape)
display(risk_table.head())

Risk table generated successfully! Shape: (292, 16)


,source,destination,source_center,destination_center,leg_count,trip_count,distance_km,median_actual_time,pred_ratio,lo50,hi50,lo90,hi90,pred_excess_h,excess_lo90_h,excess_hi90_h
0,IND000000AAL,IND411033AAA,IND000000AAL,IND411033AAA,14,14,24.28745,68.5,2.5708,2.5208,2.6208,2.4208,2.7208,107.6021,97.3271,117.8771
1,IND000000ACA,IND144001AAB,IND000000ACA,IND144001AAB,13,13,70.06870,83.0,1.6122,1.5622,1.6622,1.4622,1.7622,50.8163,38.3663,63.2663
2,IND000000ACA,IND160002AAC,IND000000ACA,IND160002AAC,59,59,88.90970,132.0,2.0645,2.0145,2.1145,1.9145,2.2145,140.5161,120.7161,160.3161
3,IND000000ACB,IND000000AEL,IND000000ACB,IND000000AEL,17,17,61.28135,114.5,1.8321,1.7821,1.8821,1.6821,1.9821,95.2701,78.0951,112.4451
4,IND000000ACB,IND110030AAD,IND000000ACB,IND110030AAD,21,21,49.44540,88.0,1.6923,1.6423,1.7423,1.5423,1.8423,60.9231,47.7231,74.1231


In [11]:
from pathlib import Path

# Ensure the day5 experiment directory exists
out_dir = Path("../experiments/day5")
out_dir.mkdir(parents=True, exist_ok=True)

# Save the risk table
out_path = out_dir / "risk_table.csv"
risk_table.to_csv(out_path, index=False)

print(f"Risk table successfully saved to: {out_path}")

Risk table successfully saved to: ..\experiments\day5\risk_table.csv


In [12]:
import pandas as pd

# Reload the risk table for verification
risk_df = pd.read_csv("../experiments/day5/risk_table.csv")

print("Risk Ratio Statistics Summary (Pred Ratio vs Median Actual):")
print(
    risk_df[["pred_ratio", "median_actual_time", "distance_km"]]
    .describe()
    .round(4)
)

Risk Ratio Statistics Summary (Pred Ratio vs Median Actual):
       pred_ratio  median_actual_time  distance_km
count    292.0000            292.0000     292.0000
mean       2.1131            412.3527     276.9089
std        0.6660            614.8791     430.8079
min        1.0198             17.0000      10.1735
25%        1.7707             62.0000      29.7552
50%        1.9718            111.0000      57.6216
75%        2.2680            475.8750     338.2030
max        8.2925           3045.0000    2190.6314


In [13]:
import pandas as pd
from pathlib import Path

# 1. Load core train legs and weather daily data
core_train_legs = pd.read_csv("../data/processed/core_train_legs.csv")
weather_daily = pd.read_csv("../data/processed/weather_daily.csv")

print("Core train legs shape:", core_train_legs.shape)
print("Weather daily shape:", weather_daily.shape)

# 2. Preview columns to verify join keys (e.g., hub/center and date)
print("Core train columns:", core_train_legs.columns.tolist()[:6])
print("Weather daily columns:", weather_daily.columns.tolist()[:6])

Core train legs shape: (5395, 15)
Weather daily shape: (2470, 11)
Core train columns: ['trip_uuid', 'source_center', 'destination_center', 'od_start_time', 'od_end_time', 'trip_creation_time']
Weather daily columns: ['date', 'rain_mm', 'precipitation_mm', 'temperature_mean_c', 'hub', 'requested_latitude']


In [14]:
# Convert timestamps to datetime format for merging
core_train_legs["date"] = pd.to_datetime(
    core_train_legs["trip_creation_time"]
).dt.strftime("%Y-%m-%d")
weather_daily["date"] = pd.to_datetime(weather_daily["date"]).dt.strftime(
    "%Y-%m-%d"
)

# Merge weather data onto core train legs using source hub and date
merged_train = pd.merge(
    core_train_legs,
    weather_daily,
    left_on=["source_center", "date"],
    right_on=["hub", "date"],
    how="left",
)

print("Merged core train legs with weather shape:", merged_train.shape)
print("Missing rain_mm values:", merged_train["rain_mm"].isna().sum())
display(merged_train[["trip_uuid", "source_center", "date", "rain_mm"]].head())

Merged core train legs with weather shape: (5395, 26)
Missing rain_mm values: 239


,trip_uuid,source_center,date,rain_mm
0,trip-153671041653548748,IND209304AAA,2018-09-12,1.3
1,trip-153671041653548748,IND462022AAA,2018-09-12,0.3
2,trip-153671043369099517,IND000000ACB,2018-09-12,0.1
3,trip-153671043369099517,IND562132AAA,2018-09-12,2.6
4,trip-153671046011330457,IND400072AAB,2018-09-12,0.6


In [19]:
import numpy as np
from sklearn.metrics import root_mean_squared_error

# Fill missing rain values with 0 (dry days)
merged_train["rain_mm"] = merged_train["rain_mm"].fillna(0)

# Check if target delay ratio column exists (or compute it)
# Let's inspect available columns to ensure we use the correct target name
print(
    "Target / delay columns available:",
    [c for c in merged_train.columns if "ratio" in c or "delay" in c],
)

Target / delay columns available: ['delay_ratio']


In [17]:
print(merged_train.columns.tolist())

['trip_uuid', 'source_center', 'destination_center', 'od_start_time', 'od_end_time', 'trip_creation_time', 'source_name', 'destination_name', 'route_type', 'actual_time_min', 'osrm_time_min', 'osrm_distance_km', 'scan_rows', 'delay_ratio', 'trip_created', 'date', 'rain_mm', 'precipitation_mm', 'temperature_mean_c', 'hub', 'requested_latitude', 'requested_longitude', 'grid_latitude', 'grid_longitude', 'weather_model', 'timezone']


In [18]:
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor

# Fill missing values for the features and target
merged_train["rain_mm"] = merged_train["rain_mm"].fillna(0)
y = merged_train["delay_ratio"].fillna(1.0)

# Define feature sets using the exact column names available
features_base = ["osrm_distance_km", "osrm_time_min"]
features_weather = ["osrm_distance_km", "osrm_time_min", "rain_mm"]

X_base = merged_train[features_base].fillna(0)
X_weather = merged_train[features_weather].fillna(0)

# Train baseline vs weather-augmented models
rf_base = RandomForestRegressor(n_estimators=50, random_state=42)
rf_weather = RandomForestRegressor(n_estimators=50, random_state=42)

rf_base.fit(X_base, y)
rf_weather.fit(X_weather, y)

pred_base = rf_base.predict(X_base)
pred_weather = rf_weather.predict(X_weather)

rmse_base = root_mean_squared_error(y, pred_base)
rmse_weather = root_mean_squared_error(y, pred_weather)

print(f"RMSE without weather: {rmse_base:.4f}")
print(f"RMSE with weather (rain_mm): {rmse_weather:.4f}")
print(
    f"Improvement: {((rmse_base - rmse_weather) / rmse_base) * 100:.2f}% drop in error"
)

RMSE without weather: 0.7014
RMSE with weather (rain_mm): 0.6475
Improvement: 7.69% drop in error


In [20]:
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor

# Prepare features (with and without rain)
features_base = ["distance_km", "median_actual_time"]
features_weather = ["distance_km", "median_actual_time", "rain_mm"]

X_base = merged_train[features_base].fillna(0)
X_weather = merged_train[features_weather].fillna(0)
y = merged_train["delay_ratio"].fillna(1.0)

# Quick train/eval split or out-of-fold comparison
# Let's train a quick Random Forest baseline to check RMSE impact
rf_base = RandomForestRegressor(n_estimators=50, random_state=42)
rf_weather = RandomForestRegressor(n_estimators=50, random_state=42)

rf_base.fit(X_base, y)
rf_weather.fit(X_weather, y)

pred_base = rf_base.predict(X_base)
pred_weather = rf_weather.predict(X_weather)

rmse_base = root_mean_squared_error(y, pred_base)
rmse_weather = root_mean_squared_error(y, pred_weather)

print(f"RMSE without weather: {rmse_base:.4f}")
print(f"RMSE with weather (rain_mm): {rmse_weather:.4f}")
print(
    f"Improvement: {((rmse_base - rmse_weather) / rmse_base) * 100:.2f}% drop in"
    " error"
)

KeyError: "None of [Index(['distance_km', 'median_actual_time'], dtype='object')] are in the [columns]"